# 04 Embed And Pack Artifacts

Create BGE-M3 embedding JSONL artifacts, run offline retrieval smoke, and zip outputs for download.

This notebook keeps `state/`, `payloads/`, `embeddings/`, and `reports/` together so later sessions can resume from the same artifact bundle.

In [ ]:
import json, os, shutil, subprocess, sys
from pathlib import Path

CORPUS_SLUG = 'tuvi-golden-corpus'
SCRIPTS_SLUG = 'tuvi-battu-scripts'

ALL_STRATEGIES = ['chunk_fixed_512', 'chunk_structure_parent_child', 'chunk_semantic_embedding_bge_m3']
SOURCE_IDS = ['TVKL', 'TVNL', 'TVHS', 'TVGM']
RUN_TAG = 'part_a'
PARTITION_MODE = 'strategy'
SELECTED_STRATEGIES = ['chunk_fixed_512', 'chunk_structure_parent_child']
SELECTED_SOURCES = list(SOURCE_IDS)

CORPUS_DIR = Path(f'/kaggle/input/{CORPUS_SLUG}/benchmark/tuvi_golden_dataset')
SCRIPTS_DIR = Path(f'/kaggle/input/{SCRIPTS_SLUG}')
if not CORPUS_DIR.exists():
    CORPUS_DIR = Path.cwd() / 'benchmark' / 'tuvi_golden_dataset'
if not SCRIPTS_DIR.exists():
    SCRIPTS_DIR = Path.cwd()

if PARTITION_MODE not in {'strategy', 'source'}:
    raise ValueError('PARTITION_MODE must be strategy or source')
ACTIVE_STRATEGIES = [item for item in ALL_STRATEGIES if item in SELECTED_STRATEGIES] if PARTITION_MODE == 'strategy' else list(ALL_STRATEGIES)
ACTIVE_SOURCES = [item for item in SOURCE_IDS if item in SELECTED_SOURCES] if PARTITION_MODE == 'source' else list(SOURCE_IDS)
if not ACTIVE_STRATEGIES:
    raise ValueError('No active strategies selected')
if not ACTIVE_SOURCES:
    raise ValueError('No active sources selected')

OUTPUT_ROOT = Path('/kaggle/working/w3_local_outputs')
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
REPORTS = OUTPUT_ROOT / 'reports'
STATE = OUTPUT_ROOT / 'state'
PAYLOADS = OUTPUT_ROOT / 'payloads'
EMBEDDINGS = OUTPUT_ROOT / 'embeddings'
for path in [REPORTS, STATE, PAYLOADS, EMBEDDINGS]:
    path.mkdir(parents=True, exist_ok=True)

print('CORPUS_DIR =', CORPUS_DIR)
print('SCRIPTS_DIR =', SCRIPTS_DIR)
print('OUTPUT_ROOT =', OUTPUT_ROOT)
print(json.dumps({'run_tag': RUN_TAG, 'partition_mode': PARTITION_MODE, 'active_strategies': ACTIVE_STRATEGIES, 'active_sources': ACTIVE_SOURCES}, ensure_ascii=False, indent=2))

In [ ]:
for strategy in ACTIVE_STRATEGIES:
    for source in ACTIVE_SOURCES:
        cmd = [
            sys.executable, '-B', str(SCRIPTS_DIR / 'scripts' / 'embed_chunks.py'),
            '--chunks-input', str(OUTPUT_ROOT / 'chunks' / strategy),
            '--source-id', source,
            '--chunking-strategy', strategy,
            '--output', str(EMBEDDINGS / strategy / f'{source}_{strategy}_embeddings.jsonl'),
            '--summary-output', str(REPORTS / f'embed_{source}_{strategy}.json'),
            '--retrieval-smoke-output', str(REPORTS / f'retrieval_{source}_{strategy}.json'),
            '--state-output', str(STATE / f'{source}_{strategy}_embedding_state.json'),
            '--resume',
            '--embedding-slot', 'bge_m3',
            '--embedding-backend', 'local',
            '--model', 'BAAI/bge-m3',
            '--expected-dim', '1024',
            '--local-embedding-model', 'BAAI/bge-m3',
            '--local-embedding-batch-size', '16',
            '--vector-index-name', 'chunkVectorBgeM3',
        ]
        print('Running source/strategy =', source, strategy)
        subprocess.run(cmd, cwd=SCRIPTS_DIR, check=True)

print('Skipped strategies =', [item for item in ALL_STRATEGIES if item not in ACTIVE_STRATEGIES])
print('Skipped sources =', [item for item in SOURCE_IDS if item not in ACTIVE_SOURCES])

In [ ]:
for strategy in ACTIVE_STRATEGIES:
    for source in ACTIVE_SOURCES:
        embedding_path = EMBEDDINGS / strategy / f'{source}_{strategy}_embeddings.jsonl'
        state_path = STATE / f'{source}_{strategy}_embedding_state.json'
        print(source, strategy)
        print('  embedding =', embedding_path.exists(), embedding_path)
        print('  state     =', state_path.exists(), state_path)

archive = shutil.make_archive(f'/kaggle/working/w3_local_outputs_{RUN_TAG}', 'zip', OUTPUT_ROOT)
print('Wrote', archive)
print('Zip contains state/, payloads/, embeddings/, and reports/ for later resume/import.')